In [ ]:
from pathlib import Path
import shutil
import zipfile
import os

inputs = Path("/kaggle/input")
destination = Path("/kaggle/working/BTP")

matches = list(inputs.rglob("run_public252.py"))

if matches:
    source = matches[0].parent
    shutil.copytree(source, destination, dirs_exist_ok=True)
else:
    archives = list(inputs.rglob("*.zip"))
    code_found = False

    for archive in archives:
        with zipfile.ZipFile(archive) as z:
            if "BTP/run_public252.py" in z.namelist():
                # Extract only our code archive.
                target = Path("/kaggle/working").resolve()
                for name in z.namelist():
                    if not (target / name).resolve().is_relative_to(target):
                        raise ValueError("Unsafe archive path")
                z.extractall(target)
                code_found = True
                break

    if not code_found:
        print("Attached input folders:", list(inputs.iterdir()))
        print("ZIP files found:", archives)
        raise RuntimeError(
            "BTP code is not attached. Add the dataset containing BTP-code.zip."
        )

assert (destination / "run_public252.py").is_file()
os.chdir(destination)
print("Code ready:", os.getcwd())

In [ ]:
import importlib
from pathlib import Path

for name in [
    "torch", "torchvision", "numpy", "pandas",
    "scipy", "skimage", "PIL", "einops", "torchmetrics", "tqdm"
]:
    try:
        module = importlib.import_module(name)
        print(f"{name}: OK ({getattr(module, '__version__', 'unknown')})")
    except Exception as error:
        print(f"{name}: FAILED — {error}")

import torch
print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 1
    ), "GB")

roots = [
    p.parent
    for p in Path("/kaggle/input").rglob("train_data.csv")
    if (p.parent / "visible_28").is_dir()
    and (p.parent / "labels").is_dir()
]

print("\nPrepared dataset folders:", roots)

In [ ]:
from pathlib import Path
import subprocess
import sys

DATA_ROOT = Path("/kaggle/input/datasets/arnavnigamd/btp-data/public252")

def run(script, *args):
    subprocess.run(
        [sys.executable, script, *map(str, args)],
        cwd="/kaggle/working/BTP",
        check=True,
    )

run("verify_evaluation.py")
run("run_public252.py", "--root", DATA_ROOT)
run("verify_portability.py", "--root", DATA_ROOT)
run("verify_public252_pipeline.py", "--root", DATA_ROOT)

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
import sys
from datetime import datetime

DATA_ROOT = "/kaggle/input/datasets/arnavnigamd/btp-data/public252"
OUT = "/kaggle/working/gpu_smoke_" + datetime.now().strftime("%Y%m%d_%H%M%S")

subprocess.run(
    [
        sys.executable, "gpu_smoke.py",
        "--root", DATA_ROOT,
        "--out", OUT,
    ],
    cwd="/kaggle/working/BTP",
    check=True,
)


In [ ]:
import subprocess
import sys
from datetime import datetime

OUT = "/kaggle/working/fixed_crop_" + datetime.now().strftime("%Y%m%d_%H%M%S")

subprocess.run(
    [
        sys.executable, "diagnostic_public252.py",
        "--root", "/kaggle/input/datasets/arnavnigamd/btp-data/public252",
        "--out", OUT,
        "--transpose-image",
        "--steps", "200",
    ],
    cwd="/kaggle/working/BTP",
    check=True,
)

print("Diagnostic results:", OUT)

In [ ]:
from IPython.display import display, Image

display(Image(
    filename="/kaggle/working/fixed_crop_20260917_144904/predictions.png"
))

In [ ]:
import subprocess
import sys
from datetime import datetime

OUT = "/kaggle/working/fixed_crop_1000_" + datetime.now().strftime("%Y%m%d_%H%M%S")

subprocess.run(
    [
        sys.executable, "diagnostic_public252.py",
        "--root", "/kaggle/input/datasets/arnavnigamd/btp-data/public252",
        "--out", OUT,
        "--transpose-image",
        "--steps", "1000",
    ],
    cwd="/kaggle/working/BTP",
    check=True,
)

print("Results:", OUT)

In [ ]:
from pathlib import Path
from IPython.display import display, Image
import json

run_dir = Path("/kaggle/working/fixed_crop_1000_20260917_145850")

metrics = json.loads((run_dir / "metrics.json").read_text())
print("Final step:", metrics[-1]["step"])
print("Final mIoU:", metrics[-1]["foreground_miou_present"])

display(Image(filename=str(run_dir / "predictions.png")))

In [ ]:
from pathlib import Path
import shutil
from IPython.display import FileLink, display

run_dir = Path("/kaggle/working/fixed_crop_1000_20260917_145850")
assert run_dir.is_dir(), "Run folder not found"

archive = shutil.make_archive(
    "/kaggle/working/fixed_crop_1000_backup",
    "zip",
    root_dir=run_dir.parent,
    base_dir=run_dir.name,
)

display(FileLink(archive))

##version 2

In [ ]:
from pathlib import Path
import shutil, zipfile, os

inputs = Path('/kaggle/input')
working = Path('/kaggle/working')

def safe_extract(archive, target):
    target.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            if not (target / name).resolve().is_relative_to(target.resolve()):
                raise ValueError('Unsafe ZIP member')
        z.extractall(target)

code = list(inputs.rglob('compare_diagnostic_lr.py'))
if not code:
    for archive in inputs.rglob('*.zip'):
        with zipfile.ZipFile(archive) as z:
            is_code = 'BTP/compare_diagnostic_lr.py' in z.namelist()
        if is_code:
            target = working / 'lr_code_unpacked'
            safe_extract(archive, target)
            code.extend(target.rglob('compare_diagnostic_lr.py'))
assert len(code) == 1, f'Expected one updated code input: {code}'
shutil.copytree(code[0].parent, working/'BTP', dirs_exist_ok=True)
os.chdir(working/'BTP')

run_name = 'fixed_crop_1000_20260917_145850'
runs = [p.parent for p in inputs.rglob('diagnostic_checkpoint.pt')
        if p.parent.name == run_name]
if not runs:
    for archive in inputs.rglob('*.zip'):
        with zipfile.ZipFile(archive) as z:
            is_run = f'{run_name}/diagnostic_checkpoint.pt' in z.namelist()
        if is_run:
            target = working / 'reference_unpacked'
            safe_extract(archive, target)
            runs.append(target/run_name)
assert len(runs) == 1, f'Expected one saved 1,000-step run: {runs}'
REFERENCE_RUN = runs[0]
DATA_ROOT = Path('/kaggle/input/datasets/arnavnigamd/btp-data/public252')
assert (DATA_ROOT/'train_data.csv').is_file()
print('Code ready; reference:', REFERENCE_RUN)

In [ ]:
import subprocess
import sys
import torch
from pathlib import Path
from datetime import datetime

assert torch.cuda.is_available(), "Enable GPU T4 ×2 first"

DATA_ROOT = Path("/kaggle/input/datasets/arnavnigamd/btp-data/public252")
REFERENCE_RUN = Path(
    "/kaggle/input/datasets/arnavnigamd/checkpoint1/"
    "fixed_crop_1000_20260917_145850"
)
OUT = Path(
    "/kaggle/working/lr_comparison_"
    + datetime.now().strftime("%Y%m%d_%H%M%S")
)

subprocess.run(
    [sys.executable, "verify_diagnostic_comparison.py"],
    cwd="/kaggle/working/BTP",
    check=True,
)

subprocess.run(
    [
        sys.executable, "compare_diagnostic_lr.py",
        "--root", str(DATA_ROOT),
        "--from-run", str(REFERENCE_RUN),
        "--out", str(OUT),
        "--steps", "500",
        "--eval-every", "10",
    ],
    cwd="/kaggle/working/BTP",
    check=True,
)

print("Results:", OUT)
print((OUT / "comparison_summary.json").read_text())

In [ ]:
import subprocess
import sys
from datetime import datetime

checkpoint = (
    "/kaggle/working/lr_comparison_20260917_160952/"
    "control_4e-4/diagnostic_checkpoint.pt"
)

subprocess.run(
    [
        sys.executable, "test.py",
        "--data_root",
        "/kaggle/input/datasets/arnavnigamd/btp-data/public252",
        "--transpose_image",
        "--eval_split", "val",
        "--limit", "1",
        "--pretrained_model_path", checkpoint,
        "--outf", "/kaggle/working/evaluations",
        "--name", "full_frame_smoke_" + datetime.now().strftime("%H%M%S"),
    ],
    cwd="/kaggle/working/BTP",
    check=True,
)

In [ ]:
from pathlib import Path
from IPython.display import display, Image

comparison = Path("/kaggle/working/lr_comparison_20260917_160952")

for arm in ["control_4e-4", "low_4e-5"]:
    path = comparison / arm / "best_predictions.png"
    print(arm)
    if path.is_file():
        display(Image(filename=str(path)))
    else:
        print("Restore the saved comparison folder and update its path.")

In [ ]:
from pathlib import Path

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    print(f"\nSearching {root}")
    for pattern in ["best_predictions.png", "comparison_summary.json", "*.zip"]:
        for path in root.rglob(pattern):
            print(path)

In [ ]:
from pathlib import Path
import shutil
import zipfile

inputs = Path("/kaggle/input")
target = Path("/kaggle/working/BTP")

# Find an already-extracted copy of the updated code.
sources = [
    p.parent for p in inputs.rglob("run_public252.py")
    if "--stop-after-epoch" in p.read_text()
]

if sources:
    shutil.copytree(sources[0], target, dirs_exist_ok=True)
else:
    candidates = []
    for archive in inputs.rglob("*.zip"):
        if not zipfile.is_zipfile(archive):
            continue
        with zipfile.ZipFile(archive) as z:
            name = "BTP/run_public252.py"
            if name in z.namelist():
                if b"--stop-after-epoch" in z.read(name):
                    candidates.append(archive)

    if not candidates:
        raise FileNotFoundError(
            "Attach BTP-code-pilot.zip through Add Input, then rerun this cell."
        )

    with zipfile.ZipFile(candidates[0]) as z:
        destination = Path("/kaggle/working").resolve()
        for member in z.infolist():
            resolved = (destination / member.filename).resolve()
            if not resolved.is_relative_to(destination):
                raise ValueError("Unsafe archive path")
        z.extractall(destination)

assert (target / "run_public252.py").is_file()
assert "--stop-after-epoch" in (target / "run_public252.py").read_text()
print("Updated pilot code ready:", target)

In [ ]:
from pathlib import Path
import subprocess, sys, torch
CODE = Path('/kaggle/working/BTP')
DATA = Path('/kaggle/input/datasets/arnavnigamd/btp-data/public252')
assert CODE.joinpath('run_public252.py').is_file(), 'Extract the updated BTP-code-pilot.zip first.'
assert '--stop-after-epoch' in CODE.joinpath('run_public252.py').read_text(), 'Use the updated pilot code ZIP.'
assert torch.cuda.is_available(), 'Enable the GPU before starting.'
print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'run_public252.py', '--root', str(DATA), '--epochs', '500', '--stop-after-epoch', '3'], cwd=CODE, check=True)


In [ ]:
from datetime import datetime
import shutil, zipfile
from IPython.display import display, FileLink
NAME = 'baseline_pilot_' + datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT = Path('/kaggle/working/baseline_runs')
RUN = OUTPUT / NAME
try:
    subprocess.run([sys.executable, '-u', 'run_public252.py', '--root', str(DATA),
        '--train', '--epochs', '500', '--stop-after-epoch', '3', '--batch-size', '1',
        '--workers', '0', '--output', str(OUTPUT), '--name', NAME], cwd=CODE, check=True)
finally:
    if RUN.exists():
        archive = shutil.make_archive(str(Path('/kaggle/working') / NAME), 'zip', root_dir=OUTPUT, base_dir=NAME)
        with zipfile.ZipFile(archive) as z:
            assert z.testzip() is None, 'Archive integrity check failed'
            required = [NAME + '/model/' + f for f in ['last.pt', 'best_iou.pth', 'best_psnr.pth']]
            missing = [f for f in required if f not in z.namelist()]
        print('Archive:', archive, 'bytes:', Path(archive).stat().st_size)
        print('Missing checkpoint files:', missing)
        display(FileLink(archive))
        print('Download this archive before stopping the session. If checkpoint files are missing, report the training error.')


In [ ]:
from pathlib import Path
import subprocess, sys, torch
CODE = Path('/kaggle/working/BTP')
DATA = Path('/kaggle/input/datasets/arnavnigamd/btp-data/public252')
assert CODE.joinpath('run_public252.py').is_file(), 'Extract the updated BTP-code-pilot.zip first.'
assert '--stop-after-epoch' in CODE.joinpath('run_public252.py').read_text(), 'Use the updated pilot code ZIP.'
assert torch.cuda.is_available(), 'Enable the GPU before starting.'
print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'run_public252.py', '--root', str(DATA), '--epochs', '500', '--stop-after-epoch', '3'], cwd=CODE, check=True)


In [ ]:
# Locate the complete saved epoch-3 model folder.
import zipfile
expected = 'baseline_pilot_20260917_190720'
refs = list(Path('/kaggle/input').rglob(expected + '/model/last.pt'))
if not refs:
    for archive in Path('/kaggle/input').rglob('*.zip'):
        if not zipfile.is_zipfile(archive):
            continue
        with zipfile.ZipFile(archive) as z:
            if expected + '/model/last.pt' not in z.namelist():
                continue
            dest = Path('/kaggle/working/restored_pilot').resolve()
            for member in z.infolist():
                if not (dest / member.filename).resolve().is_relative_to(dest):
                    raise ValueError('Unsafe archive path')
            z.extractall(dest)
            refs = [dest / expected / 'model/last.pt']
            break
assert len(refs) == 1, f'Attach exactly one saved epoch-3 pilot. Found: {refs}'
REFERENCE = refs[0]
for name in ['last.pt', 'best_iou.pth', 'best_psnr.pth']:
    assert REFERENCE.with_name(name).is_file(), f'Missing {name}'
print('Resume checkpoint:', REFERENCE)

from datetime import datetime
import shutil, zipfile
from IPython.display import display, FileLink
NAME = 'baseline_to_epoch10_' + datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT = Path('/kaggle/working/baseline_runs')
RUN = OUTPUT / NAME
try:
    subprocess.run([sys.executable, '-u', 'run_public252.py', '--root', str(DATA),
        '--train', '--epochs', '500', '--stop-after-epoch', '10', '--resume', str(REFERENCE), '--batch-size', '1',
        '--workers', '0', '--output', str(OUTPUT), '--name', NAME], cwd=CODE, check=True)
finally:
    if RUN.exists():
        archive = shutil.make_archive(str(Path('/kaggle/working') / NAME), 'zip', root_dir=OUTPUT, base_dir=NAME)
        with zipfile.ZipFile(archive) as z:
            assert z.testzip() is None, 'Archive integrity check failed'
            required = [NAME + '/model/' + f for f in ['last.pt', 'best_iou.pth', 'best_psnr.pth']]
            missing = [f for f in required if f not in z.namelist()]
        print('Archive:', archive, 'bytes:', Path(archive).stat().st_size)
        print('Missing checkpoint files:', missing)
        display(FileLink(archive))
        print('Download this archive before stopping the session. If checkpoint files are missing, report the training error.')


In [ ]:
from pathlib import Path
import subprocess, sys, torch
CODE = Path('/kaggle/working/BTP')
DATA = Path('/kaggle/input/datasets/arnavnigamd/btp-data/public252')
assert CODE.joinpath('run_public252.py').is_file(), 'Extract the updated BTP-code-pilot.zip first.'
assert '--stop-after-epoch' in CODE.joinpath('run_public252.py').read_text(), 'Use the updated pilot code ZIP.'
assert torch.cuda.is_available(), 'Enable the GPU before starting.'
print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'run_public252.py', '--root', str(DATA), '--epochs', '500', '--stop-after-epoch', '3'], cwd=CODE, check=True)


In [ ]:
# Locate the complete saved epoch-10 model folder.
import zipfile
expected = 'baseline_to_epoch10_20260917_195856'
refs = list(Path('/kaggle/input').rglob(expected + '/model/last.pt'))
if not refs:
    for archive in Path('/kaggle/input').rglob('*.zip'):
        if not zipfile.is_zipfile(archive):
            continue
        with zipfile.ZipFile(archive) as z:
            if expected + '/model/last.pt' not in z.namelist():
                continue
            dest = Path('/kaggle/working/restored_pilot').resolve()
            for member in z.infolist():
                if not (dest / member.filename).resolve().is_relative_to(dest):
                    raise ValueError('Unsafe archive path')
            z.extractall(dest)
            refs = [dest / expected / 'model/last.pt']
            break
assert len(refs) == 1, f'Attach exactly one saved epoch-10 pilot. Found: {refs}'
REFERENCE = refs[0]
for name in ['last.pt', 'best_iou.pth', 'best_psnr.pth']:
    assert REFERENCE.with_name(name).is_file(), f'Missing {name}'
print('Resume checkpoint:', REFERENCE)

from datetime import datetime
import shutil, zipfile
from IPython.display import display, FileLink
NAME = 'baseline_to_epoch50_' + datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT = Path('/kaggle/working/baseline_runs')
RUN = OUTPUT / NAME
try:
    subprocess.run([sys.executable, '-u', 'run_public252.py', '--root', str(DATA),
        '--train', '--epochs', '500', '--stop-after-epoch', '50', '--resume', str(REFERENCE), '--batch-size', '1',
        '--workers', '0', '--output', str(OUTPUT), '--name', NAME], cwd=CODE, check=True)
finally:
    if RUN.exists():
        archive = shutil.make_archive(str(Path('/kaggle/working') / NAME), 'zip', root_dir=OUTPUT, base_dir=NAME)
        with zipfile.ZipFile(archive) as z:
            assert z.testzip() is None, 'Archive integrity check failed'
            required = [NAME + '/model/' + f for f in ['last.pt', 'best_iou.pth', 'best_psnr.pth']]
            missing = [f for f in required if f not in z.namelist()]
        print('Archive:', archive, 'bytes:', Path(archive).stat().st_size)
        print('Missing checkpoint files:', missing)
        display(FileLink(archive))
        print('Download this archive before stopping the session. If checkpoint files are missing, report the training error.')


In [ ]:
import json
for path in sorted((RUN / 'result').glob('*_training.json')):
    print(json.dumps(json.loads(path.read_text()), indent=2))
print('Download the complete archive, then share these metrics and the per-class segmentation CSVs.')


In [2]:
from pathlib import Path
import shutil
import zipfile

inputs = Path("/kaggle/input")
destination = Path("/kaggle/working").resolve()
code = destination / "BTP"

sources = [
    p.parent for p in inputs.rglob("run_public252.py")
    if "--stop-after-epoch" in p.read_text()
]

if sources:
    shutil.copytree(sources[0], code, dirs_exist_ok=True)
else:
    found = False
    for archive in inputs.rglob("*.zip"):
        if not zipfile.is_zipfile(archive):
            continue
        with zipfile.ZipFile(archive) as z:
            entry = "BTP/run_public252.py"
            if entry not in z.namelist():
                continue
            if b"--stop-after-epoch" not in z.read(entry):
                continue
            for member in z.infolist():
                if not (destination / member.filename).resolve().is_relative_to(destination):
                    raise ValueError("Unsafe archive path")
            z.extractall(destination)
            found = True
            break

    assert found, "Attach BTP-code-pilot.zip through Add Input, then rerun."

assert (code / "test.py").is_file()
print("Code ready:", code)

Code ready: /kaggle/working/BTP


In [3]:
from pathlib import Path
import subprocess, sys, torch
CODE = Path('/kaggle/working/BTP')
DATA = Path('/kaggle/input/datasets/arnavnigamd/btp-data/public252')
assert CODE.joinpath('run_public252.py').is_file(), 'Extract the updated BTP-code-pilot.zip first.'
assert '--stop-after-epoch' in CODE.joinpath('run_public252.py').read_text(), 'Use the updated pilot code ZIP.'
assert torch.cuda.is_available(), 'Enable the GPU before starting.'
print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'run_public252.py', '--root', str(DATA), '--epochs', '500', '--stop-after-epoch', '3'], cwd=CODE, check=True)


Tesla T4
{
  "protocol": "public252-v1",
  "protocol_sha256": "77e00da7d2ee3dc6509e840e1dd4139fd07d338fbecfab083ec28c48e36d7318",
  "preflight": "passed",
  "splits": {
    "train": 202,
    "val": 25,
    "test": 25
  },
  "metrics": {
    "reference_amplitude": 1.0,
    "label": "PSNR_ref1 / SSIM_ref1",
    "meaning": "Fixed reference scale, not an assertion that all target values lie in [0,1]. No per-image inferred range."
  },
  "training_command": [
    "/usr/bin/python3",
    "train.py",
    "--data_root",
    "/kaggle/input/datasets/arnavnigamd/btp-data/public252/",
    "--transpose_image",
    "--batch_size",
    "1",
    "--max_epoch",
    "500",
    "--name",
    "public252_v1_baseline",
    "--workers",
    "0",
    "--outf",
    "/kaggle/working/BTP/exp/CRSDUN/",
    "--stop_after_epoch",
    "3"
  ],
  "note": "Header/membership checks plus one training-scene loader check; full pixel audit is recorded separately."
}


CompletedProcess(args=['/usr/bin/python3', 'run_public252.py', '--root', '/kaggle/input/datasets/arnavnigamd/btp-data/public252', '--epochs', '500', '--stop-after-epoch', '3'], returncode=0)

In [4]:
# Locate the complete saved epoch-50 model folder.
import zipfile
expected = 'baseline_to_epoch50_20260917_204827'
refs = list(Path('/kaggle/input').rglob(expected + '/model/last.pt'))
if not refs:
    for archive in Path('/kaggle/input').rglob('*.zip'):
        if not zipfile.is_zipfile(archive):
            continue
        with zipfile.ZipFile(archive) as z:
            if expected + '/model/last.pt' not in z.namelist():
                continue
            dest = Path('/kaggle/working/restored_pilot').resolve()
            for member in z.infolist():
                if not (dest / member.filename).resolve().is_relative_to(dest):
                    raise ValueError('Unsafe archive path')
            z.extractall(dest)
            refs = [dest / expected / 'model/last.pt']
            break
assert len(refs) == 1, f'Attach exactly one saved epoch-50 pilot. Found: {refs}'
REFERENCE = refs[0]
for name in ['last.pt', 'best_iou.pth', 'best_psnr.pth']:
    assert REFERENCE.with_name(name).is_file(), f'Missing {name}'
print('Resume checkpoint:', REFERENCE)



Resume checkpoint: /kaggle/input/datasets/arnavnigamd/checkpoint4/baseline_to_epoch50_20260917_204827/model/last.pt


In [5]:
from datetime import datetime
from IPython.display import display, FileLink
import shutil
OUT = Path('/kaggle/working') / ('checkpoint_audit_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
OUT.mkdir()
try:
    for name, weights in [('best_iou_a', 'best_iou.pth'), ('best_iou_b', 'best_iou.pth'), ('last_epoch50', 'last.pt')]:
        subprocess.run([sys.executable, '-u', 'test.py', '--data_root', str(DATA),
            '--transpose_image', '--eval_split', 'val', '--pretrained_model_path', str(REFERENCE.with_name(weights)),
            '--outf', str(OUT), '--name', name], cwd=CODE, check=True)
finally:
    archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT.parent, base_dir=OUT.name)
    display(FileLink(archive))
    print('Download this evaluation archive for comparison:', archive)


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


load model from /kaggle/input/datasets/arnavnigamd/checkpoint4/baseline_to_epoch50_20260917_204827/model/best_iou.pth
Evaluated 1/25
Evaluated 2/25
Evaluated 3/25
Evaluated 4/25
Evaluated 5/25
Evaluated 6/25
Evaluated 7/25
Evaluated 8/25
Evaluated 9/25
Evaluated 10/25
Evaluated 11/25
Evaluated 12/25
Evaluated 13/25
Evaluated 14/25
Evaluated 15/25
Evaluated 16/25
Evaluated 17/25
Evaluated 18/25
Evaluated 19/25
Evaluated 20/25
Evaluated 21/25
Evaluated 22/25
Evaluated 23/25
Evaluated 24/25
Evaluated 25/25
Completed val: 25 scenes. Results: /kaggle/working/checkpoint_audit_20260918_063350/best_iou_a/evaluation_val


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


load model from /kaggle/input/datasets/arnavnigamd/checkpoint4/baseline_to_epoch50_20260917_204827/model/best_iou.pth
Evaluated 1/25
Evaluated 2/25
Evaluated 3/25
Evaluated 4/25
Evaluated 5/25
Evaluated 6/25
Evaluated 7/25
Evaluated 8/25
Evaluated 9/25
Evaluated 10/25
Evaluated 11/25
Evaluated 12/25
Evaluated 13/25
Evaluated 14/25
Evaluated 15/25
Evaluated 16/25
Evaluated 17/25
Evaluated 18/25
Evaluated 19/25
Evaluated 20/25
Evaluated 21/25
Evaluated 22/25
Evaluated 23/25
Evaluated 24/25
Evaluated 25/25
Completed val: 25 scenes. Results: /kaggle/working/checkpoint_audit_20260918_063350/best_iou_b/evaluation_val


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


load model from /kaggle/input/datasets/arnavnigamd/checkpoint4/baseline_to_epoch50_20260917_204827/model/last.pt
Evaluated 1/25
Evaluated 2/25
Evaluated 3/25
Evaluated 4/25
Evaluated 5/25
Evaluated 6/25
Evaluated 7/25
Evaluated 8/25
Evaluated 9/25
Evaluated 10/25
Evaluated 11/25
Evaluated 12/25
Evaluated 13/25
Evaluated 14/25
Evaluated 15/25
Evaluated 16/25
Evaluated 17/25
Evaluated 18/25
Evaluated 19/25
Evaluated 20/25
Evaluated 21/25
Evaluated 22/25
Evaluated 23/25
Evaluated 24/25
Evaluated 25/25
Completed val: 25 scenes. Results: /kaggle/working/checkpoint_audit_20260918_063350/last_epoch50/evaluation_val


/kaggle/working/checkpoint_audit_20260918_063350.zip

Download this evaluation archive for comparison: /kaggle/working/checkpoint_audit_20260918_063350.zip


In [6]:
from pathlib import Path
import shutil, zipfile, subprocess, sys
INPUTS = Path('/kaggle/input')
CODE = Path('/kaggle/working/BTP-paired')
def safe_extract(z, dest):
    dest = dest.resolve()
    for member in z.infolist():
        if not (dest / member.filename).resolve().is_relative_to(dest):
            raise ValueError('Unsafe ZIP member')
    z.extractall(dest)
if not CODE.exists():
    sources = list(INPUTS.rglob('paired_full_training.py'))
    if len(sources) == 1:
        shutil.copytree(sources[0].parent, CODE)
    else:
        found = []
        for archive in INPUTS.rglob('*.zip'):
            if zipfile.is_zipfile(archive):
                with zipfile.ZipFile(archive) as z:
                    if 'BTP/paired_full_training.py' in z.namelist(): found.append(archive)
        assert len(found) == 1, 'Attach exactly one BTP-code-paired-full.zip input.'
        staging = Path('/kaggle/working/paired_code_extract')
        staging.mkdir(exist_ok=False)
        with zipfile.ZipFile(found[0]) as z: safe_extract(z, staging)
        shutil.copytree(staging / 'BTP', CODE)
assert (CODE / 'paired_full_training.py').is_file()
expected = 'baseline_to_epoch50_20260917_204827'
refs = list(INPUTS.rglob(expected + '/model/last.pt'))
if not refs:
    for archive in INPUTS.rglob('*.zip'):
        if not zipfile.is_zipfile(archive): continue
        with zipfile.ZipFile(archive) as z:
            if expected + '/model/last.pt' in z.namelist():
                dest = Path('/kaggle/working/paired_reference')
                safe_extract(z, dest)
                refs = [dest / expected / 'model/last.pt']
                break
assert len(refs) == 1, 'Attach the complete epoch50 run ZIP or extracted folder.'
REFERENCE = refs[0].parent.parent
DATA = Path('/kaggle/input/datasets/arnavnigamd/btp-data/public252')
import torch
assert torch.cuda.is_available(), 'Enable the GPU.'
print('Code:', CODE, 'Reference:', REFERENCE, 'GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'verify_paired_full_training.py'], cwd=CODE, check=True)


Code: /kaggle/working/BTP-paired Reference: /kaggle/input/datasets/arnavnigamd/checkpoint4/baseline_to_epoch50_20260917_204827 GPU: Tesla T4
PASS: original state unchanged; model/Adam moments/scaler/RNG retained; quarter LR preserved through epochs51-60; original scheduler horizon retained.


CompletedProcess(args=['/usr/bin/python3', 'verify_paired_full_training.py'], returncode=0)

In [7]:
from datetime import datetime
from IPython.display import display, FileLink
OUT = Path('/kaggle/working') / ('paired_full_lr_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
try:
    subprocess.run([sys.executable, '-u', 'paired_full_training.py', '--root', str(DATA),
        '--reference', str(REFERENCE), '--out', str(OUT)], cwd=CODE, check=True)
finally:
    if OUT.exists():
        archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT.parent, base_dir=OUT.name)
        with zipfile.ZipFile(archive) as z:
            assert z.testzip() is None
            missing = [f'{OUT.name}/{arm}/model/{name}' for arm in ['control','quarter_lr']
                for name in ['last.pt','best_iou.pth','best_psnr.pth']
                if f'{OUT.name}/{arm}/model/{name}' not in z.namelist()]
        print('Archive:', archive, 'Missing checkpoint files:', missing)
        display(FileLink(archive))
        print('Download the complete ZIP before closing the session. Check logs for successful epoch60 completion in BOTH arms.')


{
  "protocol": "public252-v1",
  "protocol_sha256": "77e00da7d2ee3dc6509e840e1dd4139fd07d338fbecfab083ec28c48e36d7318",
  "preflight": "passed",
  "splits": {
    "train": 202,
    "val": 25,
    "test": 25
  },
  "metrics": {
    "reference_amplitude": 1.0,
    "label": "PSNR_ref1 / SSIM_ref1",
    "meaning": "Fixed reference scale, not an assertion that all target values lie in [0,1]. No per-image inferred range."
  },
  "training_command": [
    "/usr/bin/python3",
    "train.py",
    "--data_root",
    "/kaggle/input/datasets/arnavnigamd/btp-data/public252/",
    "--transpose_image",
    "--batch_size",
    "1",
    "--max_epoch",
    "500",
    "--name",
    "public252_v1_baseline",
    "--workers",
    "0",
    "--outf",
    "/kaggle/working/BTP-paired/exp/CRSDUN/"
  ],
  "note": "Header/membership checks plus one training-scene loader check; full pixel audit is recorded separately."
}
Namespace(gpu_id='0', data_root='/kaggle/input/datasets/arnavnigamd/btp-data/public252/', mask

2026-09-18 06:52:09,841 - INFO: Resuming after epoch 50


Epoch 51/500 | Batch 10/202 | Rec 0.005144 | Seg 1.021869 | Elapsed 0.5m | ETA 10.0m
Epoch 51/500 | Batch 20/202 | Rec 0.004673 | Seg 0.885248 | Elapsed 0.8m | ETA 7.1m
Epoch 51/500 | Batch 30/202 | Rec 0.004890 | Seg 0.889467 | Elapsed 1.0m | ETA 6.0m
Epoch 51/500 | Batch 40/202 | Rec 0.004708 | Seg 0.830084 | Elapsed 1.3m | ETA 5.3m
Epoch 51/500 | Batch 50/202 | Rec 0.004741 | Seg 0.851144 | Elapsed 1.6m | ETA 4.8m
Epoch 51/500 | Batch 60/202 | Rec 0.004777 | Seg 0.856618 | Elapsed 1.9m | ETA 4.4m
Epoch 51/500 | Batch 70/202 | Rec 0.004640 | Seg 0.841192 | Elapsed 2.1m | ETA 4.0m
Epoch 51/500 | Batch 80/202 | Rec 0.004523 | Seg 0.864833 | Elapsed 2.4m | ETA 3.7m
Epoch 51/500 | Batch 90/202 | Rec 0.004381 | Seg 0.855603 | Elapsed 2.7m | ETA 3.4m
Epoch 51/500 | Batch 100/202 | Rec 0.004258 | Seg 0.863140 | Elapsed 3.0m | ETA 3.0m
Epoch 51/500 | Batch 110/202 | Rec 0.004150 | Seg 0.853932 | Elapsed 3.3m | ETA 2.7m
Epoch 51/500 | Batch 120/202 | Rec 0.004001 | Seg 0.831384 | Elapsed 3.5m

2026-09-18 06:58:03,869 - INFO: Rec loss: 0.0034556188257549437
2026-09-18 06:58:03,869 - INFO: Seg loss: 0.7952528654938877
2026-09-18 06:58:03,869 - INFO: λ_rec: 1
2026-09-18 06:58:03,869 - INFO: λ_seg: 0.0001
2026-09-18 06:59:20,060 - INFO: -------------------Epoch: 51------------------------
2026-09-18 06:59:20,069 - INFO: 
Validation stats:
       PSNR      SSIM       MSE
0  30.93045  0.932048  0.000824
2026-09-18 06:59:20,077 - INFO: 
Validation stats:
                   IoU        F1      Prec    recall       Acc
bg,           0.992691  0.996282  0.994796  0.997873  0.993005
real potato   0.652244  0.789475  0.752484  0.830400  0.999470
fake potato   0.168151  0.287866  0.936563  0.170088  0.999387
real apple    0.242740  0.390607  0.300350  0.558602  0.993988
fake apple    0.343274  0.511059  0.869586  0.361906  0.997386
real orange   0.568682  0.724995  0.720526  0.729620  0.999271
fake orange   0.630840  0.773591  0.995721  0.632555  0.998940
real grape    0.324315  0.489743 

Epoch 52/500 | Batch 10/202 | Rec 0.004387 | Seg 0.879511 | Elapsed 0.2m | ETA 4.4m
Epoch 52/500 | Batch 20/202 | Rec 0.003697 | Seg 0.813333 | Elapsed 0.5m | ETA 4.1m
Epoch 52/500 | Batch 30/202 | Rec 0.003997 | Seg 0.810980 | Elapsed 0.7m | ETA 3.9m
Epoch 52/500 | Batch 40/202 | Rec 0.003910 | Seg 0.857997 | Elapsed 0.9m | ETA 3.7m
Epoch 52/500 | Batch 50/202 | Rec 0.003744 | Seg 0.853588 | Elapsed 1.1m | ETA 3.5m
Epoch 52/500 | Batch 60/202 | Rec 0.003558 | Seg 0.832673 | Elapsed 1.4m | ETA 3.2m
Epoch 52/500 | Batch 70/202 | Rec 0.003600 | Seg 0.833755 | Elapsed 1.6m | ETA 3.0m
Epoch 52/500 | Batch 80/202 | Rec 0.003506 | Seg 0.808088 | Elapsed 1.8m | ETA 2.8m
Epoch 52/500 | Batch 90/202 | Rec 0.003450 | Seg 0.815087 | Elapsed 2.0m | ETA 2.5m
Epoch 52/500 | Batch 100/202 | Rec 0.003327 | Seg 0.798685 | Elapsed 2.3m | ETA 2.3m
Epoch 52/500 | Batch 110/202 | Rec 0.003268 | Seg 0.807645 | Elapsed 2.5m | ETA 2.1m
Epoch 52/500 | Batch 120/202 | Rec 0.003229 | Seg 0.806077 | Elapsed 2.7m 

...

Epoch 60/500 | Batch 10/202 | Rec 0.002350 | Seg 0.639390 | Elapsed 0.2m | ETA 4.4m
Epoch 60/500 | Batch 20/202 | Rec 0.001818 | Seg 0.551259 | Elapsed 0.5m | ETA 4.1m
Epoch 60/500 | Batch 30/202 | Rec 0.001717 | Seg 0.535893 | Elapsed 0.7m | ETA 3.9m
Epoch 60/500 | Batch 40/202 | Rec 0.001722 | Seg 0.569842 | Elapsed 0.9m | ETA 3.7m
Epoch 60/500 | Batch 50/202 | Rec 0.001745 | Seg 0.572813 | Elapsed 1.1m | ETA 3.4m
Epoch 60/500 | Batch 60/202 | Rec 0.001746 | Seg 0.584075 | Elapsed 1.4m | ETA 3.2m
Epoch 60/500 | Batch 70/202 | Rec 0.001786 | Seg 0.589153 | Elapsed 1.6m | ETA 3.0m
Epoch 60/500 | Batch 80/202 | Rec 0.001802 | Seg 0.612224 | Elapsed 1.8m | ETA 2.7m
Epoch 60/500 | Batch 90/202 | Rec 0.001827 | Seg 0.615209 | Elapsed 2.0m | ETA 2.5m
Epoch 60/500 | Batch 100/202 | Rec 0.001823 | Seg 0.610932 | Elapsed 2.2m | ETA 2.3m
Epoch 60/500 | Batch 110/202 | Rec 0.001799 | Seg 0.607558 | Elapsed 2.5m | ETA 2.1m
Epoch 60/500 | Batch 120/202 | Rec 0.001774 | Seg 0.607915 | Elapsed 2.7m 

2026-09-18 08:50:32,586 - INFO: Rec loss: 0.0017787577131403482
2026-09-18 08:50:32,586 - INFO: Seg loss: 0.6258107938683859
2026-09-18 08:50:32,586 - INFO: λ_rec: 1
2026-09-18 08:50:32,587 - INFO: λ_seg: 0.0001
2026-09-18 08:51:55,423 - INFO: -------------------Epoch: 60------------------------
2026-09-18 08:51:55,427 - INFO: 
Validation stats:
       PSNR      SSIM       MSE
0  35.05889  0.960359  0.000326
2026-09-18 08:51:55,435 - INFO: 
Validation stats:
                   IoU        F1      Prec    recall       Acc
bg,           0.993679  0.996779  0.994700  0.998968  0.993950
real potato   0.441795  0.612798  0.999711  0.441852  0.999332
fake potato   0.855886  0.922298  1.000000  0.855886  0.999895
real apple    0.610643  0.758210  0.732737  0.785626  0.998272
fake apple    0.632230  0.774633  0.857648  0.706353  0.998449
real orange   0.488310  0.656150  0.999053  0.488536  0.999325
fake orange   0.738719  0.849678  0.780884  0.931884  0.999056
real grape    0.301170  0.462882 

Stopped after epoch 60; scheduler horizon remains 500
{
  "experiment": {
    "reference_checkpoint_sha256": "cf6d3d5328cb1fb31997f7531cecfbf333a7fcc766c1c4102b0006d6021c0919",
    "source_epoch": 50,
    "end_epoch": 60,
    "epochs_per_arm": 10,
    "initial_metrics": {
      "iou": 0.22774086842170374,
      "psnr": 30.42753318786621
    },
    "factors": {
      "control": 1.0,
      "quarter_lr": 0.25
    },
    "best_selection_scope": "Epoch50 initial weights and epochs51-60; historical epoch49 excluded",
    "change": "Scale optimizer LR and remaining cosine schedule, including eta_min; retain moments, scaler, RNG and model",
    "limitations": "Single seed, validation-only diagnostic; numerical hardware nondeterminism remains possible",
    "source_sha256": {
      "paired_full_training.py": "c524877aac898c4d9f6bced63d804d4d3fbbae4440a782598cbdf8e37ce2d2e7",
      "train.py": "c6cb7e98dd9a55e22a20b05d1ea271b86bc874099d6295578c4fcc41bb5917f3",
      "training_state.py": "ae508e1

/kaggle/working/paired_full_lr_20260918_065140.zip

Download the complete ZIP before closing the session. Check logs for successful epoch60 completion in BOTH arms.


In [8]:
import json
summary = json.loads((OUT / 'comparison_summary.json').read_text())
for arm, result in summary['arms'].items():
    print(arm, json.dumps({k:v for k,v in result.items() if k != 'epochs'}, indent=2))
assert set(summary['arms']) == {'control', 'quarter_lr'}, 'Comparison is incomplete.'


control {
  "final": {
    "epoch": 60,
    "scheduler_horizon": 500,
    "learning_rate": 0.00038644818693541307,
    "attempted_updates": 202,
    "optimizer_updates": 202,
    "amp_skipped_updates": 0,
    "train_reconstruction_loss": 0.001971663857607337,
    "train_segmentation_loss": 0.6370254272576605,
    "val_foreground_miou_all22": 0.3976762041058326,
    "val_psnr_ref1": 34.670361022949216,
    "elapsed_seconds": 355.1619760990143,
    "peak_cuda_allocated_gib": 7.506626129150391
  },
  "best_logged_continuation_miou": 0.5050588718936055,
  "mean_last5_miou": 0.43189472687145575,
  "std_last5_miou": 0.07089516818835125,
  "actual_updates": 2020,
  "amp_skips": 0
}
quarter_lr {
  "final": {
    "epoch": 60,
    "scheduler_horizon": 500,
    "learning_rate": 9.661204673385327e-05,
    "attempted_updates": 202,
    "optimizer_updates": 202,
    "amp_skipped_updates": 0,
    "train_reconstruction_loss": 0.0017787577131403482,
    "train_segmentation_loss": 0.6258107938683859,
  

In [1]:
from pathlib import Path
import shutil, zipfile, subprocess, sys
INPUTS = Path('/kaggle/input')
CODE = Path('/kaggle/working/BTP-paired')
def safe_extract(z, dest):
    dest = dest.resolve()
    for member in z.infolist():
        if not (dest / member.filename).resolve().is_relative_to(dest):
            raise ValueError('Unsafe ZIP member')
    z.extractall(dest)
if not CODE.exists():
    sources = list(INPUTS.rglob('paired_full_training.py'))
    if len(sources) == 1:
        shutil.copytree(sources[0].parent, CODE)
    else:
        found = []
        for archive in INPUTS.rglob('*.zip'):
            if zipfile.is_zipfile(archive):
                with zipfile.ZipFile(archive) as z:
                    if 'BTP/paired_full_training.py' in z.namelist(): found.append(archive)
        assert len(found) == 1, 'Attach exactly one BTP-code-paired-full.zip input.'
        staging = Path('/kaggle/working/paired_code_extract')
        staging.mkdir(exist_ok=False)
        with zipfile.ZipFile(found[0]) as z: safe_extract(z, staging)
        shutil.copytree(staging / 'BTP', CODE)
assert (CODE / 'paired_full_training.py').is_file()
import json, hashlib, math
import torch
expected = 'paired_full_lr_20260918_065140'
relative = expected + '/quarter_lr/model/last.pt'
refs = list(INPUTS.rglob(relative))
if not refs:
    working = Path('/kaggle/working') / relative
    if working.is_file(): refs = [working]
if not refs:
    dest = Path('/kaggle/working/restored_quarter60')
    restored = dest / relative
    if restored.is_file(): refs = [restored]
    else:
        for archive in INPUTS.rglob('*.zip'):
            if not zipfile.is_zipfile(archive): continue
            with zipfile.ZipFile(archive) as z:
                if relative in z.namelist():
                    safe_extract(z, dest)
                    refs = [dest / relative]
                    break
assert len(refs) == 1, 'Attach paired_full_lr_20260918_065140.zip or its extracted folder.'
CHECKPOINT = refs[0]
for name in ['last.pt', 'best_iou.pth', 'best_psnr.pth']:
    assert CHECKPOINT.with_name(name).is_file(), f'Missing {name}'
state = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
assert state['epoch'] == 60
expected_config = dict(learning_rate=0.0001, max_epoch=500, batch_size=1,
                       workers=0, seed=3407, transpose_image=True,
                       method='CRSDUN', input_setting='Y', input_mask='SSR')
for key, value in expected_config.items():
    assert state['config'][key] == value, f'Unexpected checkpoint setting: {key}'
assert state['scheduler']['last_epoch'] == 60
assert state['scheduler']['T_max'] == 500
assert math.isclose(state['scheduler']['eta_min'], 2.5e-7)
print('Resuming quarter-rate checkpoint:', CHECKPOINT)
print('Saved next-epoch LR:', state['optimizer']['param_groups'][0]['lr'])
print('Saved best metrics:', state['best'])
del state
DATA = Path('/kaggle/input/datasets/arnavnigamd/btp-data/public252')
assert torch.cuda.is_available(), 'Enable the GPU before training.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'run_public252.py', '--root', str(DATA)], cwd=CODE, check=True)
print('Preflight only above; the next cell resumes the quarter schedule, not the printed default schedule.')


Resuming quarter-rate checkpoint: /kaggle/input/datasets/arnavnigamd/checkpoint5/paired_full_lr_20260918_065140/quarter_lr/model/last.pt
Saved next-epoch LR: 9.649760223367654e-05
Saved best metrics: {'iou': 0.5179274164506568, 'psnr': 35.058889923095705}
GPU: Tesla T4
{
  "protocol": "public252-v1",
  "protocol_sha256": "77e00da7d2ee3dc6509e840e1dd4139fd07d338fbecfab083ec28c48e36d7318",
  "preflight": "passed",
  "splits": {
    "train": 202,
    "val": 25,
    "test": 25
  },
  "metrics": {
    "reference_amplitude": 1.0,
    "label": "PSNR_ref1 / SSIM_ref1",
    "meaning": "Fixed reference scale, not an assertion that all target values lie in [0,1]. No per-image inferred range."
  },
  "training_command": [
    "/usr/bin/python3",
    "train.py",
    "--data_root",
    "/kaggle/input/datasets/arnavnigamd/btp-data/public252/",
    "--transpose_image",
    "--batch_size",
    "1",
    "--max_epoch",
    "500",
    "--name",
    "public252_v1_baseline",
    "--workers",
    "0",
    "-

In [2]:
from datetime import datetime
from IPython.display import display, FileLink
NAME = 'quarter_lr_to_epoch100_' + datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = Path('/kaggle/working') / NAME
OUT.mkdir(exist_ok=False)
shutil.copy2(CODE / 'public252-v1.json', OUT / 'dataset_protocol.json')
provenance = {
    'reference_checkpoint_sha256': hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest(),
    'source_epoch': 60, 'stop_after_epoch': 100, 'arm': 'quarter_lr',
    'note': 'Restore saved Adam/scaler/RNG and quarter-scaled500-epoch scheduler without rescaling again.',
    'source_sha256': {f: hashlib.sha256((CODE / f).read_bytes()).hexdigest()
                      for f in ['train.py', 'opt.py', 'training_state.py', 'dataset.py']}
}
(OUT / 'continuation.json').write_text(json.dumps(provenance, indent=2))
command = [sys.executable, '-u', 'train.py', '--data_root', str(DATA) + '/',
    '--transpose_image', '--batch_size', '1', '--workers', '0', '--seed', '3407',
    '--learning_rate', '0.0001', '--max_epoch', '500', '--stop_after_epoch', '100',
    '--resume', str(CHECKPOINT), '--outf', str(OUT.parent) + '/', '--name', NAME]
try:
    subprocess.run(command, cwd=CODE, check=True)
finally:
    archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT.parent, base_dir=OUT.name)
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None, 'ZIP integrity check failed'
        missing = [f'{NAME}/model/{f}' for f in ['last.pt','best_iou.pth','best_psnr.pth']
                   if f'{NAME}/model/{f}' not in z.namelist()]
    print('Archive:', archive, 'Missing checkpoint files:', missing)
    if (OUT / 'model/last.pt').is_file():
        saved = torch.load(OUT / 'model/last.pt', map_location='cpu', weights_only=True)
        print('Saved epoch:', saved['epoch'], 'Best metrics:', saved['best'])
        del saved
    display(FileLink(archive))
    print('Download the complete archive before stopping the session. Check saved epoch is100; a partial run also creates an archive.')


Namespace(gpu_id='0', data_root='/kaggle/input/datasets/arnavnigamd/btp-data/public252/', mask_path='mask/mask512x512.mat', transpose_image=True, outf='/kaggle/working/', name='quarter_lr_to_epoch100_20260918_094320', method='CRSDUN', pretrained_model_path=None, input_setting='Y', input_mask='SSR', batch_size=1, max_epoch=500, learning_rate=0.0001, resume='/kaggle/input/datasets/arnavnigamd/checkpoint5/paired_full_lr_20260918_065140/quarter_lr/model/last.pt', workers=0, seed=3407, eval_split='val', stop_after_epoch=100)
Dataset size (num. batches) 202 25


2026-09-18 09:43:31,818 - INFO: Resuming after epoch 60


Epoch 61/500 | Batch 10/202 | Rec 0.001518 | Seg 0.635539 | Elapsed 0.6m | ETA 10.9m
Epoch 61/500 | Batch 20/202 | Rec 0.001780 | Seg 0.703889 | Elapsed 0.9m | ETA 8.1m
Epoch 61/500 | Batch 30/202 | Rec 0.001944 | Seg 0.751850 | Elapsed 1.2m | ETA 6.8m
Epoch 61/500 | Batch 40/202 | Rec 0.001903 | Seg 0.740136 | Elapsed 1.5m | ETA 6.0m
Epoch 61/500 | Batch 50/202 | Rec 0.001873 | Seg 0.732524 | Elapsed 1.8m | ETA 5.5m
Epoch 61/500 | Batch 60/202 | Rec 0.001822 | Seg 0.702144 | Elapsed 2.1m | ETA 4.9m
Epoch 61/500 | Batch 70/202 | Rec 0.001788 | Seg 0.686741 | Elapsed 2.4m | ETA 4.5m
Epoch 61/500 | Batch 80/202 | Rec 0.001779 | Seg 0.668721 | Elapsed 2.7m | ETA 4.1m
Epoch 61/500 | Batch 90/202 | Rec 0.001784 | Seg 0.659866 | Elapsed 3.0m | ETA 3.8m
Epoch 61/500 | Batch 100/202 | Rec 0.001771 | Seg 0.648677 | Elapsed 3.4m | ETA 3.4m
Epoch 61/500 | Batch 110/202 | Rec 0.001767 | Seg 0.634659 | Elapsed 3.7m | ETA 3.1m
Epoch 61/500 | Batch 120/202 | Rec 0.001766 | Seg 0.639307 | Elapsed 4.0m

2026-09-18 09:50:11,183 - INFO: Rec loss: 0.0017327694095542084
2026-09-18 09:50:11,184 - INFO: Seg loss: 0.6146652539177696
2026-09-18 09:50:11,184 - INFO: λ_rec: 1
2026-09-18 09:50:11,184 - INFO: λ_seg: 0.0001
2026-09-18 09:51:45,409 - INFO: -------------------Epoch: 61------------------------
2026-09-18 09:51:45,418 - INFO: 
Validation stats:
        PSNR      SSIM       MSE
0  35.084801  0.961238  0.000324
2026-09-18 09:51:45,424 - INFO: 
Validation stats:
                   IoU        F1      Prec    recall       Acc
bg,           0.992464  0.996168  0.995480  0.996957  0.992793
real potato   0.447078  0.617862  0.986849  0.449758  0.999334
fake potato   0.554346  0.713239  0.999622  0.554462  0.999675
real apple    0.481207  0.649700  0.636425  0.663644  0.997532
fake apple    0.628104  0.771527  0.781820  0.761599  0.998297
real orange   0.698988  0.822780  0.974485  0.712019  0.999596
fake orange   0.694963  0.819984  0.734509  0.928099  0.998834
real grape    0.359669  0.52900

Epoch 62/500 | Batch 10/202 | Rec 0.001663 | Seg 0.425814 | Elapsed 0.2m | ETA 4.2m
Epoch 62/500 | Batch 20/202 | Rec 0.001528 | Seg 0.468899 | Elapsed 0.4m | ETA 4.0m
Epoch 62/500 | Batch 30/202 | Rec 0.001611 | Seg 0.543481 | Elapsed 0.7m | ETA 3.8m
Epoch 62/500 | Batch 40/202 | Rec 0.001568 | Seg 0.547406 | Elapsed 0.9m | ETA 3.6m
Epoch 62/500 | Batch 50/202 | Rec 0.001558 | Seg 0.552067 | Elapsed 1.1m | ETA 3.3m
Epoch 62/500 | Batch 60/202 | Rec 0.001585 | Seg 0.566237 | Elapsed 1.3m | ETA 3.1m
Epoch 62/500 | Batch 70/202 | Rec 0.001617 | Seg 0.579023 | Elapsed 1.5m | ETA 2.9m
Epoch 62/500 | Batch 80/202 | Rec 0.001605 | Seg 0.574908 | Elapsed 1.7m | ETA 2.7m
Epoch 62/500 | Batch 90/202 | Rec 0.001692 | Seg 0.598830 | Elapsed 2.0m | ETA 2.4m
Epoch 62/500 | Batch 100/202 | Rec 0.001652 | Seg 0.582809 | Elapsed 2.2m | ETA 2.2m
Epoch 62/500 | Batch 110/202 | Rec 0.001632 | Seg 0.582626 | Elapsed 2.4m | ETA 2.0m
Epoch 62/500 | Batch 120/202 | Rec 0.001605 | Seg 0.575617 | Elapsed 2.6m 

...

Epoch 100/500 | Batch 10/202 | Rec 0.001084 | Seg 0.380877 | Elapsed 0.2m | ETA 4.2m
Epoch 100/500 | Batch 20/202 | Rec 0.001245 | Seg 0.438558 | Elapsed 0.4m | ETA 4.0m
Epoch 100/500 | Batch 30/202 | Rec 0.001216 | Seg 0.431283 | Elapsed 0.7m | ETA 3.8m
Epoch 100/500 | Batch 40/202 | Rec 0.001214 | Seg 0.460978 | Elapsed 0.9m | ETA 3.5m
Epoch 100/500 | Batch 50/202 | Rec 0.001253 | Seg 0.455526 | Elapsed 1.1m | ETA 3.3m
Epoch 100/500 | Batch 60/202 | Rec 0.001248 | Seg 0.470468 | Elapsed 1.3m | ETA 3.1m
Epoch 100/500 | Batch 70/202 | Rec 0.001267 | Seg 0.469394 | Elapsed 1.5m | ETA 2.9m
Epoch 100/500 | Batch 80/202 | Rec 0.001260 | Seg 0.461224 | Elapsed 1.7m | ETA 2.7m
Epoch 100/500 | Batch 90/202 | Rec 0.001281 | Seg 0.472260 | Elapsed 2.0m | ETA 2.4m
Epoch 100/500 | Batch 100/202 | Rec 0.001277 | Seg 0.465369 | Elapsed 2.2m | ETA 2.2m
Epoch 100/500 | Batch 110/202 | Rec 0.001274 | Seg 0.469240 | Elapsed 2.4m | ETA 2.0m
Epoch 100/500 | Batch 120/202 | Rec 0.001273 | Seg 0.473893 | E

2026-09-18 13:33:25,675 - INFO: Rec loss: 0.0012492657512873596
2026-09-18 13:33:25,675 - INFO: Seg loss: 0.4785040553732969
2026-09-18 13:33:25,675 - INFO: λ_rec: 1
2026-09-18 13:33:25,675 - INFO: λ_seg: 0.0001
2026-09-18 13:34:44,883 - INFO: -------------------Epoch: 100------------------------
2026-09-18 13:34:44,887 - INFO: 
Validation stats:
        PSNR      SSIM       MSE
0  35.849943  0.965706  0.000268
2026-09-18 13:34:44,894 - INFO: 
Validation stats:
                   IoU        F1      Prec    recall       Acc
bg,           0.993618  0.996749  0.994402  0.999207  0.993890
real potato   0.387625  0.558648  0.990584  0.389059  0.999265
fake potato   0.949651  0.974125  0.980480  0.967951  0.999963
real apple    0.588820  0.741154  0.768745  0.715568  0.998276
fake apple    0.577017  0.731735  0.935624  0.600873  0.998337
real orange   0.463998  0.633832  0.884328  0.493979  0.999248
fake orange   0.710761  0.830881  0.732655  0.959652  0.998882
real grape    0.176948  0.3006

Stopped after epoch 100; scheduler horizon remains 500
Archive: /kaggle/working/quarter_lr_to_epoch100_20260918_094320.zip Missing checkpoint files: []
Saved epoch: 100 Best metrics: {'iou': 0.5795132702150417, 'psnr': 36.20094161987305}


/kaggle/working/quarter_lr_to_epoch100_20260918_094320.zip

Download the complete archive before stopping the session. Check saved epoch is100; a partial run also creates an archive.


In [3]:
records = [json.loads(p.read_text()) for p in sorted((OUT / 'result').glob('*_training.json'))]
assert [r['epoch'] for r in records] == list(range(61,101)), 'Training did not finish all40 epochs.'
print('Final epoch:', json.dumps(records[-1], indent=2))
print('Best mIoU among new epochs:', max(records, key=lambda r: r['val_foreground_miou_all22']))
print('AMP skips in new epochs:', sum(r['amp_skipped_updates'] for r in records))

Final epoch: {
  "epoch": 100,
  "scheduler_horizon": 500,
  "learning_rate": 9.065812145110052e-05,
  "attempted_updates": 202,
  "optimizer_updates": 202,
  "amp_skipped_updates": 0,
  "train_reconstruction_loss": 0.0012492657512873596,
  "train_segmentation_loss": 0.4785040553732969,
  "val_foreground_miou_all22": 0.5313312177334463,
  "val_psnr_ref1": 35.84994338989258,
  "elapsed_seconds": 344.1486585140228,
  "peak_cuda_allocated_gib": 7.510776519775391
}
Best mIoU among new epochs: {'epoch': 96, 'scheduler_horizon': 500, 'learning_rate': 9.137564364194375e-05, 'attempted_updates': 202, 'optimizer_updates': 202, 'amp_skipped_updates': 0, 'train_reconstruction_loss': 0.0013205602155343657, 'train_segmentation_loss': 0.4865771540910891, 'val_foreground_miou_all22': 0.5795132702150417, 'val_psnr_ref1': 35.41128036499023, 'elapsed_seconds': 346.2433886528015, 'peak_cuda_allocated_gib': 7.510776519775391}
AMP skips in new epochs: 2


In [4]:
from pathlib import Path
import os
import shutil
import zipfile
from IPython.display import display, FileLink

working = Path("/kaggle/working")
runs = [
    p for p in working.glob("quarter_lr_to_epoch100_*")
    if p.is_dir() and (p / "model/last.pt").is_file()
]
assert runs, "No saved run found. Keep this session open and share the output."

run = max(runs, key=lambda p: (p / "model/last.pt").stat().st_mtime)
archive = run.with_suffix(".zip")

# Recreate the archive from the saved run.
shutil.make_archive(
    str(run), "zip", root_dir=run.parent, base_dir=run.name
)

with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None, "Archive integrity check failed"

print("ZIP:", archive)
print(f"Size: {archive.stat().st_size / 1024**2:.1f} MiB")

os.chdir(working)
display(FileLink(archive.name))

ZIP: /kaggle/working/quarter_lr_to_epoch100_20260918_094320.zip
Size: 118.8 MiB


/kaggle/working/quarter_lr_to_epoch100_20260918_094320.zip

In [6]:
from pathlib import Path
import shutil, zipfile, subprocess, sys
INPUTS = Path('/kaggle/input')
CODE = Path('/kaggle/working/BTP-weak-audit')
def safe_extract(z, dest):
    dest = dest.resolve()
    for member in z.infolist():
        if not (dest / member.filename).resolve().is_relative_to(dest):
            raise ValueError('Unsafe ZIP member')
    z.extractall(dest)
# Identify this evaluator by its actual feature, not a shared filename.
import tempfile
marker = "parser.add_argument('--save_segmentation'"
def is_audit_source(path):
    return path.is_file() and marker in path.read_text()
if not is_audit_source(CODE / 'test.py'):
    sources = [p.parent for p in INPUTS.rglob('test.py') if is_audit_source(p)]
    if sources:
        shutil.copytree(sorted(sources)[0], CODE, dirs_exist_ok=True)
    else:
        matches = []
        archives = list(INPUTS.rglob('*.zip'))
        for archive in archives:
            if not zipfile.is_zipfile(archive): continue
            with zipfile.ZipFile(archive) as z:
                for member in z.namelist():
                    if member.endswith('/test.py') and marker.encode() in z.read(member):
                        matches.append((archive, member))
        if not matches:
            print('ZIP inputs found:', [str(p) for p in archives])
            raise FileNotFoundError('Updated audit code is not attached. Add BTP-code-weak-audit.zip, then rerun this cell.')
        archive, member = sorted(matches, key=lambda x: str(x[0]))[0]
        print('Using audit code:', archive)
        with tempfile.TemporaryDirectory(prefix='weak_audit_code_', dir='/kaggle/working') as tmp:
            staging = Path(tmp)
            with zipfile.ZipFile(archive) as z: safe_extract(z, staging)
            shutil.copytree((staging / member).parent, CODE, dirs_exist_ok=True)
assert is_audit_source(CODE / 'test.py'), 'Audit evaluator not found after extraction.'
import json, hashlib, math
import torch
expected = 'quarter_lr_to_epoch100_20260918_094320'
relative = expected + '/model/last.pt'
refs = list(INPUTS.rglob(relative))
if not refs:
    working = Path('/kaggle/working') / relative
    if working.is_file(): refs = [working]
if not refs:
    dest = Path('/kaggle/working/restored_quarter60')
    restored = dest / relative
    if restored.is_file(): refs = [restored]
    else:
        for archive in INPUTS.rglob('*.zip'):
            if not zipfile.is_zipfile(archive): continue
            with zipfile.ZipFile(archive) as z:
                if relative in z.namelist():
                    safe_extract(z, dest)
                    refs = [dest / relative]
                    break
assert len(refs) == 1, 'Attach quarter_lr_to_epoch100_20260918_094320.zip or its extracted folder.'
CHECKPOINT = refs[0]
for name in ['last.pt', 'best_iou.pth', 'best_psnr.pth']:
    assert CHECKPOINT.with_name(name).is_file(), f'Missing {name}'
CHECKPOINT = CHECKPOINT.with_name('best_iou.pth')
assert '--save_segmentation' in (CODE / 'test.py').read_text(), 'Attach the updated BTP-code-weak-audit.zip.'
DATA = Path('/kaggle/input/datasets/arnavnigamd/btp-data/public252')
assert torch.cuda.is_available(), 'Enable the GPU.'
print('Evaluating saved best segmentation checkpoint:', CHECKPOINT)


Evaluating saved best segmentation checkpoint: /kaggle/input/datasets/arnavnigamd/checkpoint6/quarter_lr_to_epoch100_20260918_094320/model/best_iou.pth


In [7]:
from datetime import datetime
from IPython.display import display, FileLink
NAME = 'weak_class_audit_' + datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = Path('/kaggle/working') / NAME
OUT.mkdir(exist_ok=False)
try:
    subprocess.run([sys.executable, '-u', 'test.py', '--data_root', str(DATA),
        '--transpose_image', '--eval_split', 'val', '--save_segmentation',
        '--pretrained_model_path', str(CHECKPOINT), '--outf', str(OUT), '--name', 'best_epoch96'], cwd=CODE, check=True)
finally:
    archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT.parent, base_dir=NAME)
    with zipfile.ZipFile(archive) as z: assert z.testzip() is None
    import os
    os.chdir('/kaggle/working')
    print('Download:', archive)
    display(FileLink(Path(archive).name))


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


load model from /kaggle/input/datasets/arnavnigamd/checkpoint6/quarter_lr_to_epoch100_20260918_094320/model/best_iou.pth
Evaluated 1/25
Evaluated 2/25
Evaluated 3/25
Evaluated 4/25
Evaluated 5/25
Evaluated 6/25
Evaluated 7/25
Evaluated 8/25
Evaluated 9/25
Evaluated 10/25
Evaluated 11/25
Evaluated 12/25
Evaluated 13/25
Evaluated 14/25
Evaluated 15/25
Evaluated 16/25
Evaluated 17/25
Evaluated 18/25
Evaluated 19/25
Evaluated 20/25
Evaluated 21/25
Evaluated 22/25
Evaluated 23/25
Evaluated 24/25
Evaluated 25/25
Completed val: 25 scenes. Results: /kaggle/working/weak_class_audit_20260918_135320/best_epoch96/evaluation_val
Download: /kaggle/working/weak_class_audit_20260918_135320.zip


/kaggle/working/weak_class_audit_20260918_135320.zip

In [8]:
import numpy as np
import sys
sys.path.insert(0,str(CODE))
from dataset import get_class_names
folder = OUT / 'best_epoch96/evaluation_val'
meta = json.loads((folder / 'evaluation.json').read_text())
assert meta['status']=='completed' and meta['scene_count']==25
hist = np.load(folder / 'confusion_matrix.npy')
classes = get_class_names()
for k in [9,12,17,22]:
    total = hist[k].sum()
    print(classes[k], 'ground-truth pixels:', int(total))
    print([(classes[j], round(100*hist[k,j]/total,2)) for j in np.argsort(hist[k])[::-1][:5]])
print('Share the downloaded ZIP; it contains all25 previews and per-scene confusion matrices, without reconstructed HSI cubes.')


real lemon ground-truth pixels: 7702
[('fake banana', np.float64(32.2)), ('real lemon', np.float64(27.53)), ('fake unknown', np.float64(20.66)), ('bg,', np.float64(13.48)), ('fake grape', np.float64(5.36))]
fake avocado ground-truth pixels: 7842
[('fake unknown', np.float64(45.75)), ('fake avocado', np.float64(18.95)), ('fake plant', np.float64(17.79)), ('bg,', np.float64(13.34)), ('real plant', np.float64(4.17))]
real banana ground-truth pixels: 14726
[('fake banana', np.float64(42.65)), ('real lemon', np.float64(28.45)), ('bg,', np.float64(26.97)), ('fake unknown', np.float64(0.75)), ('fake pepper', np.float64(0.7))]
fake unknown ground-truth pixels: 5522
[('fake unknown', np.float64(35.08)), ('bg,', np.float64(21.46)), ('fake orange', np.float64(12.12)), ('fake lemon', np.float64(10.32)), ('fake onion', np.float64(6.81))]
Share the downloaded ZIP; it contains all25 previews and per-scene confusion matrices, without reconstructed HSI cubes.


In [10]:
from pathlib import Path
import shutil, zipfile, subprocess, sys
INPUTS = Path('/kaggle/input')
CODE = Path('/kaggle/working/BTP-training-diagnostic')
def safe_extract(z, dest):
    dest = dest.resolve()
    for member in z.infolist():
        if not (dest / member.filename).resolve().is_relative_to(dest):
            raise ValueError('Unsafe ZIP member')
    z.extractall(dest)
# Identify this evaluator by its actual feature, not a shared filename.
import tempfile
marker = 'Training-only inference diagnostic; fixed class-rich crops'
def is_audit_source(path):
    return path.is_file() and marker in path.read_text()
if not is_audit_source(CODE / 'diagnose_training_classes.py'):
    sources = [p.parent for p in INPUTS.rglob('diagnose_training_classes.py') if is_audit_source(p)]
    if sources:
        shutil.copytree(sorted(sources)[0], CODE, dirs_exist_ok=True)
    else:
        matches = []
        archives = list(INPUTS.rglob('*.zip'))
        for archive in archives:
            if not zipfile.is_zipfile(archive): continue
            with zipfile.ZipFile(archive) as z:
                for member in z.namelist():
                    if member.endswith('/diagnose_training_classes.py') and marker.encode() in z.read(member):
                        matches.append((archive, member))
        if not matches:
            print('ZIP inputs found:', [str(p) for p in archives])
            raise FileNotFoundError('Updated audit code is not attached. Add BTP-code-training-diagnostic.zip, then rerun this cell.')
        archive, member = sorted(matches, key=lambda x: str(x[0]))[0]
        print('Using audit code:', archive)
        with tempfile.TemporaryDirectory(prefix='weak_audit_code_', dir='/kaggle/working') as tmp:
            staging = Path(tmp)
            with zipfile.ZipFile(archive) as z: safe_extract(z, staging)
            shutil.copytree((staging / member).parent, CODE, dirs_exist_ok=True)
assert is_audit_source(CODE / 'diagnose_training_classes.py'), 'Audit evaluator not found after extraction.'
import json, hashlib, math
import torch
expected = 'quarter_lr_to_epoch100_20260918_094320'
relative = expected + '/model/last.pt'
refs = list(INPUTS.rglob(relative))
if not refs:
    working = Path('/kaggle/working') / relative
    if working.is_file(): refs = [working]
if not refs:
    dest = Path('/kaggle/working/restored_quarter60')
    restored = dest / relative
    if restored.is_file(): refs = [restored]
    else:
        for archive in INPUTS.rglob('*.zip'):
            if not zipfile.is_zipfile(archive): continue
            with zipfile.ZipFile(archive) as z:
                if relative in z.namelist():
                    safe_extract(z, dest)
                    refs = [dest / relative]
                    break
assert len(refs) == 1, 'Attach quarter_lr_to_epoch100_20260918_094320.zip or its extracted folder.'
CHECKPOINT = refs[0]
for name in ['last.pt', 'best_iou.pth', 'best_psnr.pth']:
    assert CHECKPOINT.with_name(name).is_file(), f'Missing {name}'
CHECKPOINT = CHECKPOINT.with_name('best_iou.pth')
assert (CODE / 'diagnose_training_classes.py').is_file()
DATA = Path('/kaggle/input/datasets/arnavnigamd/btp-data/public252')
assert torch.cuda.is_available(), 'Enable the GPU.'
print('Evaluating saved best segmentation checkpoint:', CHECKPOINT)


Evaluating saved best segmentation checkpoint: /kaggle/input/datasets/arnavnigamd/checkpoint6/quarter_lr_to_epoch100_20260918_094320/model/best_iou.pth


In [11]:
from datetime import datetime
from IPython.display import display, FileLink
NAME = 'training_class_diagnostic_' + datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = Path('/kaggle/working') / NAME
try:
    subprocess.run([sys.executable, '-u', 'diagnose_training_classes.py', '--root', str(DATA),
        '--checkpoint', str(CHECKPOINT), '--out', str(OUT)], cwd=CODE, check=True)
finally:
    if OUT.exists():
        archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT.parent, base_dir=NAME)
        with zipfile.ZipFile(archive) as z: assert z.testzip() is None
        import os
        os.chdir('/kaggle/working')
        print('Download:', archive)
        display(FileLink(Path(archive).name))


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


load model from /kaggle/input/datasets/arnavnigamd/checkpoint6/quarter_lr_to_epoch100_20260918_094320/model/best_iou.pth
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Completed training diagnostic 1/8
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Completed training diagnostic 2/8
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Completed training diagnostic 3/8
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Completed training diagnostic 4/8
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Completed training diagnostic 5/8
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Completed training diagnostic 6/8
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Completed training diagnostic 7/8
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Evaluated 1/1
Completed training diagnostic 8/8
Download: /kaggle/working/training_class_diagnostic_20260918_140423.zip


/kaggle/working/training_class_diagnostic_20260918_140423.zip

In [12]:
record = json.loads((OUT / 'diagnostic.json').read_text())
assert record.get('status') == 'completed', 'Diagnostic did not finish.'
for row in json.loads((OUT / 'summary.json').read_text()):
    print(row['class_id'],row['scene'],row['mode'],
          'IoU:',round(100*row['target_iou'],2),'recall:',round(100*row['target_recall'],2))
print('These are selected TRAINING-scene diagnostics, not validation or test scores.')


9 2021-11-05_013 full IoU: 83.84 recall: 83.84
9 2021-11-05_013 full_roi IoU: 83.84 recall: 83.84
9 2021-11-05_013 crop_registered IoU: 85.24 recall: 85.4
9 2021-11-05_013 crop_training_mask IoU: 86.0 recall: 86.13
9 2021-11-05_005 full IoU: 80.77 recall: 80.77
9 2021-11-05_005 full_roi IoU: 80.77 recall: 80.77
9 2021-11-05_005 crop_registered IoU: 75.34 recall: 75.34
9 2021-11-05_005 crop_training_mask IoU: 64.65 recall: 64.65
12 2021-11-04_059 full IoU: 5.64 recall: 5.64
12 2021-11-04_059 full_roi IoU: 5.65 recall: 5.65
12 2021-11-04_059 crop_registered IoU: 24.38 recall: 24.38
12 2021-11-04_059 crop_training_mask IoU: 25.55 recall: 25.55
12 2021-11-04_058 full IoU: 0.03 recall: 0.03
12 2021-11-04_058 full_roi IoU: 0.03 recall: 0.03
12 2021-11-04_058 crop_registered IoU: 15.11 recall: 15.11
12 2021-11-04_058 crop_training_mask IoU: 21.57 recall: 21.57
17 2021-11-10_025 full IoU: 2.79 recall: 2.8
17 2021-11-10_025 full_roi IoU: 2.79 recall: 2.8
17 2021-11-10_025 crop_registered IoU: 5

In [14]:
from pathlib import Path
import shutil, zipfile, subprocess, sys
INPUTS = Path('/kaggle/input')
CODE = Path('/kaggle/working/BTP-loss-pair')
def safe_extract(z, dest):
    dest = dest.resolve()
    for member in z.infolist():
        if not (dest / member.filename).resolve().is_relative_to(dest):
            raise ValueError('Unsafe ZIP member')
    z.extractall(dest)
# Identify this evaluator by its actual feature, not a shared filename.
import tempfile
marker = 'Matched epoch101-110 segmentation-loss experiment'
def is_audit_source(path):
    return path.is_file() and marker in path.read_text()
if not is_audit_source(CODE / 'paired_loss_weight.py'):
    sources = [p.parent for p in INPUTS.rglob('paired_loss_weight.py') if is_audit_source(p)]
    if sources:
        shutil.copytree(sorted(sources)[0], CODE, dirs_exist_ok=True)
    else:
        matches = []
        archives = list(INPUTS.rglob('*.zip'))
        for archive in archives:
            if not zipfile.is_zipfile(archive): continue
            with zipfile.ZipFile(archive) as z:
                for member in z.namelist():
                    if member.endswith('/paired_loss_weight.py') and marker.encode() in z.read(member):
                        matches.append((archive, member))
        if not matches:
            print('ZIP inputs found:', [str(p) for p in archives])
            raise FileNotFoundError('Updated audit code is not attached. Add BTP-code-loss-pair.zip, then rerun this cell.')
        archive, member = sorted(matches, key=lambda x: str(x[0]))[0]
        print('Using audit code:', archive)
        with tempfile.TemporaryDirectory(prefix='weak_audit_code_', dir='/kaggle/working') as tmp:
            staging = Path(tmp)
            with zipfile.ZipFile(archive) as z: safe_extract(z, staging)
            shutil.copytree((staging / member).parent, CODE, dirs_exist_ok=True)
assert is_audit_source(CODE / 'paired_loss_weight.py'), 'Audit evaluator not found after extraction.'
import json, hashlib, math
import torch
expected = 'quarter_lr_to_epoch100_20260918_094320'
relative = expected + '/model/last.pt'
refs = list(INPUTS.rglob(relative))
if not refs:
    working = Path('/kaggle/working') / relative
    if working.is_file(): refs = [working]
if not refs:
    dest = Path('/kaggle/working/restored_quarter60')
    restored = dest / relative
    if restored.is_file(): refs = [restored]
    else:
        for archive in INPUTS.rglob('*.zip'):
            if not zipfile.is_zipfile(archive): continue
            with zipfile.ZipFile(archive) as z:
                if relative in z.namelist():
                    safe_extract(z, dest)
                    refs = [dest / relative]
                    break
assert len(refs) == 1, 'Attach quarter_lr_to_epoch100_20260918_094320.zip or its extracted folder.'
CHECKPOINT = refs[0]
for name in ['last.pt', 'best_iou.pth', 'best_psnr.pth']:
    assert CHECKPOINT.with_name(name).is_file(), f'Missing {name}'
REFERENCE = CHECKPOINT.parent.parent
assert (CODE / 'paired_loss_weight.py').is_file()
DATA = Path('/kaggle/input/datasets/arnavnigamd/btp-data/public252')
assert torch.cuda.is_available(), 'Enable the GPU.'
print('Epoch100 reference:', REFERENCE)
subprocess.run([sys.executable,'verify_loss_pair.py'],cwd=CODE,check=True)


Epoch100 reference: /kaggle/input/datasets/arnavnigamd/checkpoint6/quarter_lr_to_epoch100_20260918_094320
PASS: both loss branches retain optimizer/scheduler/model/RNG/scaler; legacy resume accepted; incompatible loss rejected; segmentation gradient scaling verified.


CompletedProcess(args=['/usr/bin/python3', 'verify_loss_pair.py'], returncode=0)

In [15]:
from datetime import datetime
from IPython.display import display, FileLink
NAME = 'paired_loss_' + datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = Path('/kaggle/working') / NAME
try:
    subprocess.run([sys.executable, '-u', 'paired_loss_weight.py', '--root', str(DATA),
        '--reference', str(REFERENCE), '--out', str(OUT)], cwd=CODE, check=True)
finally:
    if OUT.exists():
        archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT.parent, base_dir=NAME)
        with zipfile.ZipFile(archive) as z:
            assert z.testzip() is None
            missing = [f'{NAME}/{arm}/model/{file}' for arm in ['control_seg1e-4','higher_seg1e-3']
                for file in ['last.pt','best_iou.pth','best_psnr.pth']
                if f'{NAME}/{arm}/model/{file}' not in z.namelist()]
        print('Archive:', archive, 'Missing checkpoint files:', missing)
        import os
        os.chdir('/kaggle/working')
        display(FileLink(Path(archive).name))
        print('Download before ending the session. Confirm BOTH arms reached epoch110.')


{
  "protocol": "public252-v1",
  "protocol_sha256": "77e00da7d2ee3dc6509e840e1dd4139fd07d338fbecfab083ec28c48e36d7318",
  "preflight": "passed",
  "splits": {
    "train": 202,
    "val": 25,
    "test": 25
  },
  "metrics": {
    "reference_amplitude": 1.0,
    "label": "PSNR_ref1 / SSIM_ref1",
    "meaning": "Fixed reference scale, not an assertion that all target values lie in [0,1]. No per-image inferred range."
  },
  "training_command": [
    "/usr/bin/python3",
    "train.py",
    "--data_root",
    "/kaggle/input/datasets/arnavnigamd/btp-data/public252/",
    "--transpose_image",
    "--batch_size",
    "1",
    "--max_epoch",
    "500",
    "--name",
    "public252_v1_baseline",
    "--workers",
    "0",
    "--outf",
    "/kaggle/working/BTP-loss-pair/exp/CRSDUN/"
  ],
  "note": "Header/membership checks plus one training-scene loader check; full pixel audit is recorded separately."
}
Namespace(gpu_id='0', data_root='/kaggle/input/datasets/arnavnigamd/btp-data/public252/', m

2026-09-18 14:16:21,309 - INFO: Resuming after epoch 100


Epoch 101/500 | Batch 10/202 | Rec 0.001053 | Seg 0.364604 | Elapsed 0.4m | ETA 8.2m
Epoch 101/500 | Batch 20/202 | Rec 0.001165 | Seg 0.425312 | Elapsed 0.6m | ETA 5.8m
Epoch 101/500 | Batch 30/202 | Rec 0.001143 | Seg 0.421291 | Elapsed 0.8m | ETA 4.8m
Epoch 101/500 | Batch 40/202 | Rec 0.001181 | Seg 0.429075 | Elapsed 1.1m | ETA 4.3m
Epoch 101/500 | Batch 50/202 | Rec 0.001212 | Seg 0.410835 | Elapsed 1.3m | ETA 3.9m
Epoch 101/500 | Batch 60/202 | Rec 0.001250 | Seg 0.423836 | Elapsed 1.5m | ETA 3.5m
Epoch 101/500 | Batch 70/202 | Rec 0.001230 | Seg 0.427573 | Elapsed 1.7m | ETA 3.2m
Epoch 101/500 | Batch 80/202 | Rec 0.001227 | Seg 0.426073 | Elapsed 1.9m | ETA 2.9m
Epoch 101/500 | Batch 90/202 | Rec 0.001229 | Seg 0.439549 | Elapsed 2.1m | ETA 2.7m
Epoch 101/500 | Batch 100/202 | Rec 0.001229 | Seg 0.447814 | Elapsed 2.4m | ETA 2.4m
Epoch 101/500 | Batch 110/202 | Rec 0.001230 | Seg 0.443557 | Elapsed 2.6m | ETA 2.2m
Epoch 101/500 | Batch 120/202 | Rec 0.001243 | Seg 0.442634 | E

2026-09-18 14:20:56,693 - INFO: Rec loss: 0.0012524601965070712
2026-09-18 14:20:56,693 - INFO: Seg loss: 0.4577836528936825
2026-09-18 14:20:56,693 - INFO: λ_rec: 1
2026-09-18 14:20:56,693 - INFO: λ_seg: 0.0001
2026-09-18 14:22:14,612 - INFO: -------------------Epoch: 101------------------------
2026-09-18 14:22:14,617 - INFO: 
Validation stats:
        PSNR      SSIM       MSE
0  35.368203  0.963969  0.000302
2026-09-18 14:22:14,623 - INFO: 
Validation stats:
                   IoU        F1      Prec    recall       Acc
bg,           0.994266  0.997075  0.995253  0.999004  0.994515
real potato   0.254782  0.406065  1.000000  0.254782  0.999108
fake potato   0.841918  0.914126  0.999503  0.842271  0.999885
real apple    0.559658  0.717618  0.695371  0.741442  0.997987
fake apple    0.570008  0.726073  0.930742  0.595255  0.998305
real orange   0.402071  0.573497  0.959012  0.409101  0.999198
fake orange   0.679805  0.809337  0.708832  0.943183  0.998728
real grape    0.369807  0.5398

Epoch 102/500 | Batch 10/202 | Rec 0.001286 | Seg 0.571856 | Elapsed 0.2m | ETA 4.2m
Epoch 102/500 | Batch 20/202 | Rec 0.001421 | Seg 0.576227 | Elapsed 0.4m | ETA 4.0m
Epoch 102/500 | Batch 30/202 | Rec 0.001373 | Seg 0.555798 | Elapsed 0.7m | ETA 3.8m
Epoch 102/500 | Batch 40/202 | Rec 0.001320 | Seg 0.519658 | Elapsed 0.9m | ETA 3.5m
Epoch 102/500 | Batch 50/202 | Rec 0.001310 | Seg 0.507659 | Elapsed 1.1m | ETA 3.3m
Epoch 102/500 | Batch 60/202 | Rec 0.001311 | Seg 0.527606 | Elapsed 1.3m | ETA 3.1m
Epoch 102/500 | Batch 70/202 | Rec 0.001276 | Seg 0.522052 | Elapsed 1.5m | ETA 2.9m
Epoch 102/500 | Batch 80/202 | Rec 0.001255 | Seg 0.508485 | Elapsed 1.7m | ETA 2.7m
Epoch 102/500 | Batch 90/202 | Rec 0.001230 | Seg 0.495186 | Elapsed 2.0m | ETA 2.4m
Epoch 102/500 | Batch 100/202 | Rec 0.001259 | Seg 0.490318 | Elapsed 2.2m | ETA 2.2m
Epoch 102/500 | Batch 110/202 | Rec 0.001225 | Seg 0.473513 | Elapsed 2.4m | ETA 2.0m
Epoch 102/500 | Batch 120/202 | Rec 0.001206 | Seg 0.470134 | E

...

Epoch 110/500 | Batch 10/202 | Rec 0.001453 | Seg 0.436256 | Elapsed 0.2m | ETA 4.2m
Epoch 110/500 | Batch 20/202 | Rec 0.001441 | Seg 0.359787 | Elapsed 0.4m | ETA 4.0m
Epoch 110/500 | Batch 30/202 | Rec 0.001384 | Seg 0.359779 | Elapsed 0.7m | ETA 3.8m
Epoch 110/500 | Batch 40/202 | Rec 0.001289 | Seg 0.351306 | Elapsed 0.9m | ETA 3.5m
Epoch 110/500 | Batch 50/202 | Rec 0.001212 | Seg 0.337663 | Elapsed 1.1m | ETA 3.3m
Epoch 110/500 | Batch 60/202 | Rec 0.001182 | Seg 0.331017 | Elapsed 1.3m | ETA 3.1m
Epoch 110/500 | Batch 70/202 | Rec 0.001169 | Seg 0.326512 | Elapsed 1.5m | ETA 2.9m
Epoch 110/500 | Batch 80/202 | Rec 0.001180 | Seg 0.325988 | Elapsed 1.7m | ETA 2.7m
Epoch 110/500 | Batch 90/202 | Rec 0.001177 | Seg 0.319562 | Elapsed 2.0m | ETA 2.4m
Epoch 110/500 | Batch 100/202 | Rec 0.001183 | Seg 0.322321 | Elapsed 2.2m | ETA 2.2m
Epoch 110/500 | Batch 110/202 | Rec 0.001186 | Seg 0.316461 | Elapsed 2.4m | ETA 2.0m
Epoch 110/500 | Batch 120/202 | Rec 0.001232 | Seg 0.320426 | E

2026-09-18 16:10:38,930 - INFO: Rec loss: 0.001199440279108758
2026-09-18 16:10:38,931 - INFO: Seg loss: 0.3089114911116586
2026-09-18 16:10:38,931 - INFO: λ_rec: 1
2026-09-18 16:10:38,931 - INFO: λ_seg: 0.001
2026-09-18 16:12:01,389 - INFO: -------------------Epoch: 110------------------------
2026-09-18 16:12:01,393 - INFO: 
Validation stats:
        PSNR      SSIM       MSE
0  35.844937  0.964543  0.000272
2026-09-18 16:12:01,400 - INFO: 
Validation stats:
                   IoU        F1      Prec    recall       Acc
bg,           0.993712  0.996796  0.994460  0.999244  0.993981
real potato   0.076639  0.142353  1.000000  0.076639  0.998895
fake potato   0.145580  0.254138  1.000000  0.145580  0.999378
real apple    0.449495  0.620159  0.580891  0.665237  0.997189
fake apple    0.724145  0.839955  0.837039  0.842992  0.998788
real orange   0.566886  0.723537  0.997356  0.567740  0.999428
fake orange   0.747482  0.855447  0.773328  0.957201  0.999074
real grape    0.203550  0.338201

Stopped after epoch 110; scheduler horizon remains 500
{
  "experiment": {
    "reference_checkpoint_sha256": "637a9da66937c997d335df07b7f2491e228ad2ae452966a24175b13fc22936f5",
    "source_epoch": 100,
    "end_epoch": 110,
    "epochs_per_arm": 10,
    "initial_metrics": {
      "iou": 0.5313312177334463,
      "psnr": 35.84994338989258
    },
    "weights": {
      "control_seg1e-4": 0.0001,
      "higher_seg1e-3": 0.001
    },
    "best_selection_scope": "Epoch100 initial weights and epochs101-110; historical epoch96 excluded",
    "change": "Change only lambda_seg; preserve quarter LR schedule, optimizer moments, scaler, RNG, model and sampling",
    "limitations": "Single seed, validation-only diagnostic; numerical hardware nondeterminism remains possible",
    "source_sha256": {
      "paired_loss_weight.py": "f1330d2d2a8b8d24ec2eca9e51e415de02a96c64a14658c374fcc7e3771b8d07",
      "train.py": "7e4b04e264a97a983bc3ed59049277a772347f414ba83c737fb6e05619a4f21b",
      "training_st

/kaggle/working/paired_loss_20260918_141555.zip

Download before ending the session. Confirm BOTH arms reached epoch110.


In [16]:
summary = json.loads((OUT / 'comparison_summary.json').read_text())
assert set(summary['arms']) == {'control_seg1e-4','higher_seg1e-3'}, 'Incomplete comparison'
for arm, result in summary['arms'].items():
    assert result['final']['epoch'] == 110
    print(arm, json.dumps({k:v for k,v in result.items() if k != 'epochs'}, indent=2))


control_seg1e-4 {
  "final": {
    "epoch": 110,
    "scheduler_horizon": 500,
    "lambda_rec": 1,
    "lambda_seg": 0.0001,
    "learning_rate": 8.875334012741181e-05,
    "attempted_updates": 202,
    "optimizer_updates": 202,
    "amp_skipped_updates": 0,
    "train_reconstruction_loss": 0.0011724064606103567,
    "train_segmentation_loss": 0.44373450572095297,
    "val_foreground_miou_all22": 0.5061350281587473,
    "val_psnr_ref1": 35.944615783691404,
    "elapsed_seconds": 347.6273500919342,
    "peak_cuda_allocated_gib": 7.511264801025391
  },
  "best_logged_continuation_miou": 0.5769871229537121,
  "mean_last5_miou": 0.53186823965506,
  "std_last5_miou": 0.03449801483812449,
  "actual_updates": 2019,
  "amp_skips": 1
}
higher_seg1e-3 {
  "final": {
    "epoch": 110,
    "scheduler_horizon": 500,
    "lambda_rec": 1,
    "lambda_seg": 0.001,
    "learning_rate": 8.875334012741181e-05,
    "attempted_updates": 202,
    "optimizer_updates": 202,
    "amp_skipped_updates": 0,
    